In [1]:
import grewpy
import yaml
import sys
import numpy as np

sys.path.insert(1, '/Users/madalina/Documents/M2TAL/stage/grex/grex2')
import pyximport
pyximport.install()
import grex.data
import grex.utils
import grex.features

connected to port: 62663


In [2]:
path = "/Users/madalina/Documents/M1TAL/stage-SK/Treebanks/UD_French-GSD-master"
grewpy.set_config('ud')
corpus = grewpy.Corpus(path)
draft = grewpy.CorpusDraft(corpus)

In [3]:
all_matches = corpus.search(grewpy.Request("pattern{X[upos=ADV]}"), clustering_parameter=['X.lemma'])

In [4]:
matches = {}
for key, value in all_matches.items():
    # remove those that have less than 10 occurrences
    if len(value) > 10:
        matches[key] = value

In [5]:
with open("../3. probability_matrix/patterns_adv.txt") as instream:
    config = yaml.load(instream, Loader=yaml.Loader)

templates = grex.utils.FeaturePredicate.from_config(config["templates"])
feature_predicate = grex.utils.FeaturePredicate.from_config(config["features"], templates=templates)

In [7]:
data = { k : list() for k in matches }
for adv, mts in matches.items():
    for match in mts:
        features = grex.data.extract_features(draft, match, feature_predicate)
        formatted_features = [
            f"{':'.join(k)}={v}" if not isinstance(v, set) else
            f"{':'.join(k)}={val}" for k, v in features.items() for val in (v if isinstance(v, set) else [v])
        ]
        data[adv].append(formatted_features)

In [8]:
unique_adv = sorted(set([k for k in data]))
unique_features = sorted(set([feat for _, matches in data.items() for m in matches for feat in m]))

idx2feature = {i : feat for i, feat in enumerate(unique_features) }
feature2idx = {feat : i for i, feat in idx2feature.items()}
idx2adv = {i : feat for i, feat in enumerate(unique_adv) }
adv2idx = {feat : i for i, feat in idx2adv.items()}

In [9]:
X = np.zeros((len(data.keys()), len(unique_features)))
for adv, samples in data.items():
    n_samples = len(matches)
    for m in samples:
        for feature in m:
            X[adv2idx[adv], feature2idx[feature]] += 1
    X[adv2idx[adv]] = X[adv2idx[adv]] / n_samples
print(f"{X.shape=}")

X.shape=(127, 209)


In [21]:
# check if X is a sparse matrix
import numpy as np
from scipy.sparse import issparse

issparse(X)

False

# DBSCAN clustering

In [11]:
import plotly.express as px
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_distances
import pandas as pd

# Compute the cosine distance matrix
distance_matrix = cosine_distances(X)

# Fit DBSCAN model
db = DBSCAN(eps=0.09, min_samples=2, metric='precomputed')
labels = db.fit_predict(distance_matrix)

#reduce dimensionality for visualization
from sklearn.decomposition import PCA
# Reduce dimensionality to 2D using PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Create a DataFrame for Plotly
df = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df['Label'] = labels
df['Adverb'] = unique_adv  # Assuming unique_adv is a list of adverb names

# Convert labels to categorical type
df['Label'] = df['Label'].astype(str)

# Create a scatter plot with Plotly
fig = px.scatter(df, x='PC1', y='PC2', color='Label', hover_data=['Adverb'],
                 color_discrete_sequence=px.colors.qualitative.Plotly)

fig.update_layout(
    title='DBSCAN Clustering',
    xaxis_title='Principal Component 1',
    yaxis_title='Principal Component 2',
    font=dict(size=12)
)

fig.show()

In [23]:
# see what there is in each cluster

db = DBSCAN(eps=0.09, min_samples=2, metric='precomputed')
labels = db.fit_predict(distance_matrix)
# Create a dictionary to store clusters
clusters = {}
for i, label in enumerate(labels):
    if label not in clusters:
        clusters[label] = []
    clusters[label].append(i)

# Print the contents of each cluster
for cluster, members in clusters.items():
    print(f'Cluster {cluster}:')
    for member in members:
        print(f'  {unique_adv[member]}')

Cluster -1:
  absolument
  afin
  ainsi
  avant
  beaucoup
  davantage
  environ
  hors
  loin
  lors
  quant
  si
  tant
  vis-à-vis
  vivement
Cluster 0:
  actuellement
  après
  assez
  aujourd'hui
  auparavant
  aussi
  autrement
  bientôt
  cependant
  certes
  clairement
  complètement
  depuis
  directement
  donc
  définitivement
  déjà
  désormais
  encore
  enfin
  ensemble
  ensuite
  entièrement
  essentiellement
  extrêmement
  facilement
  finalement
  fort
  fortement
  généralement
  hier
  ici
  immédiatement
  initialement
  juste
  largement
  longtemps
  là
  légèrement
  maintenant
  mal
  malheureusement
  mieux
  même
  naturellement
  notamment
  néanmoins
  officiellement
  parfaitement
  parfois
  particulièrement
  partiellement
  partout
  peu
  peut-être
  plus
  plutôt
  pourtant
  pratiquement
  presque
  principalement
  probablement
  progressivement
  rapidement
  rarement
  relativement
  respectivement
  récemment
  réellement
  régulièrement
  seule

In [24]:
#calculate dunn index

import numpy as np
from sklearn.metrics.pairwise import euclidean_distances

def delta(ck, cl):
    values = np.ones([len(ck), len(cl)])*10000
    
    for i in range(0, len(ck)):
        for j in range(0, len(cl)):
            values[i, j] = np.linalg.norm(ck[i]-cl[j])
            
    return np.min(values)
    
def big_delta(ci):
    values = np.zeros([len(ci), len(ci)])
    
    for i in range(0, len(ci)):
        for j in range(0, len(ci)):
            values[i, j] = np.linalg.norm(ci[i]-ci[j])
            
    return np.max(values)
    
def dunn(k_list):
    """ Dunn index [CVI]
    
    Parameters
    ----------
    k_list : list of np.arrays
        A list containing a numpy array for each cluster |c| = number of clusters
        c[K] is np.array([N, p]) (N : number of samples in cluster K, p : sample dimension)
    """
    deltas = np.ones([len(k_list), len(k_list)])*1000000
    big_deltas = np.zeros([len(k_list), 1])
    l_range = list(range(0, len(k_list)))
    
    for k in l_range:
        for l in (l_range[0:k]+l_range[k+1:]):
            deltas[k, l] = delta(k_list[k], k_list[l])
        
        big_deltas[k] = big_delta(k_list[k])

    di = np.min(deltas)/np.max(big_deltas)
    return di

In [25]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

# Compute the silhouette score
sil_score = silhouette_score(X, labels, metric='cosine')
print(f'Silhouette Score: {sil_score}')

# Compute the Calinski-Harabasz Index
calinski_harabasz = calinski_harabasz_score(X, labels)
print(f'Calinski-Harabasz Index: {calinski_harabasz}')

# Compute the Davies-Bouldin Index
davies_bouldin = davies_bouldin_score(X, labels)
print(f'Davies-Bouldin Index: {davies_bouldin}')

# Dunn index 
clusters_d = [X[labels == j] for j in np.unique(labels)]
dunn_index = dunn(clusters_d)
print(f'Dunn Index: {dunn_index}')

Silhouette Score: 0.16885741561705425
Calinski-Harabasz Index: 11.481160970840381
Davies-Bouldin Index: 2.3386502810135417
Dunn Index: 0.003716782593962533


In [26]:
# use t-sne for visualization
from sklearn.manifold import TSNE

# Reduce dimensionality to 2D using t-SNE
tsne = TSNE(n_components=2, random_state=42)
X_tsne = tsne.fit_transform(X)

# Create a scatter plot
# Create a DataFrame for Plotly
df = pd.DataFrame(X_tsne, columns=['t-SNE1', 't-SNE2'])
df['Label'] = labels
df['Adverb'] = unique_adv  # Assuming unique_adv is a list of adverb names

# Convert labels to categorical type
df['Label'] = df['Label'].astype(str)

# Create a scatter plot with Plotly
fig = px.scatter(df, x='t-SNE1', y='t-SNE2', color='Label', hover_data=['Adverb'],
                 color_discrete_sequence=px.colors.qualitative.Plotly)

fig.update_layout(
    title='DBSCAN Clustering',
    xaxis_title='Principal Component 1',
    yaxis_title='Principal Component 2',
    font=dict(size=12)
)

fig.show()